# Food-MNIST Dataset Analysis
This notebook fulfills the requirements for analyzing the Food-MNIST dataset, including dataset preparation, building a Decision Tree classifier, tuning hyperparameters, building a Neural Network classifier, and performing performance evaluation and comparison.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Thêm thư mục cha vào sys.path để import helpers.py
sys.path.append(os.path.abspath('..'))
from utils.helpers import stratified_split, plot_class_distribution, evaluate_model

# Đảm bảo thư mục figures tồn tại
os.makedirs('../figures', exist_ok=True)

### 2.1 Preparing the dataset
Download dataset and flatten images to 1D arrays for scikit-learn models.

In [ ]:
!git clone https://github.com/srohit0/food_mnist.git

In [ ]:
def load_food_images_from_meta(base_repo_path, split='train', target_size=(64, 64)):
    """Loads images based on meta train/test split files."""
    X, y = [], []
    meta_file = os.path.join(base_repo_path, 'meta', f'{split}.txt')
    images_dir = os.path.join(base_repo_path, 'images')
    
    with open(meta_file, 'r') as f:
        lines = f.read().splitlines()
        
    # Extract valid class names by looking at subdirectories in images/
    class_names = sorted([d for d in os.listdir(images_dir) if os.path.isdir(os.path.join(images_dir, d))])
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}
    
    for line in lines:
        if not line: continue
        class_name = line.split('/')[0]
        label = class_to_idx.get(class_name)
        if label is None: continue
        
        # Append .jpg as defined in the repo
        img_path = os.path.join(images_dir, f"{line}.jpg")
        
        try:
            # Open image, resize, convert to RGB, and flatten to 1D array
            img = Image.open(img_path).convert('RGB')
            img = img.resize(target_size)
            img_array = np.array(img).flatten()
            X.append(img_array)
            y.append(label)
        except Exception as e:
            pass
            
    return np.array(X), np.array(y), class_names

print("Loading training data...")
X_train_full, y_train_full, class_names = load_food_images_from_meta('food_mnist', split='train')
print("Loading test data...")
X_test, y_test, _ = load_food_images_from_meta('food_mnist', split='test')

print(f"Full Train shape: {X_train_full.shape}")
print(f"Test shape: {X_test.shape}")

In [ ]:
# Chia tập Train_full thành Train (80%) và Validation (20%) bằng stratified split
X_train, y_train, X_val, y_val = stratified_split(X_train_full, y_train_full, val_size=0.2, random_state=42)

print(f"Train set size: {X_train.shape[0]}")
print(f"Validation set size: {X_val.shape[0]}")

# Visualize
plot_class_distribution(y_train, title="Food-MNIST Training Distribution", save_path="../figures/food_train_dist.png")
plot_class_distribution(y_val, title="Food-MNIST Validation Distribution", save_path="../figures/food_val_dist.png")

### 2.2 Building the decision tree classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_graphviz
import graphviz

# Train using Information Gain (entropy)
dt_base = DecisionTreeClassifier(criterion='entropy', random_state=42)
dt_base.fit(X_train, y_train)

print(f"Validation accuracy of base Decision Tree: {dt_base.score(X_val, y_val):.4f}")

# Visualize the tree (limiting depth to 3 so it's readable)
dot_data = export_graphviz(dt_base, max_depth=3, feature_names=[f"px_{i}" for i in range(X_train.shape[1])],
                           class_names=class_names, filled=True, rounded=True)
graph = graphviz.Source(dot_data)
graph.render("../figures/food_tree", format="png", cleanup=True)
graph

### 2.3 Hyperparameter tuning for decision tree classifier

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Use RandomizedSearchCV to save time
random_search = RandomizedSearchCV(
    DecisionTreeClassifier(criterion='entropy', random_state=42),
    param_dist, n_iter=10, cv=3, random_state=42, n_jobs=-1, verbose=2
)
random_search.fit(X_train, y_train)

best_dt = random_search.best_estimator_
print("Best params:", random_search.best_params_)
print(f"Validation Accuracy (Tuned): {best_dt.score(X_val, y_val):.4f}")

### 2.4 Building the neural network classifier

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128),  # 2 hidden layers due to complexity of RGB images
    activation='relu',
    solver='adam',
    max_iter=300,
    early_stopping=True,
    random_state=42,
    verbose=True
)

mlp.fit(X_train, y_train)
print(f"MLP Validation Accuracy: {mlp.score(X_val, y_val):.4f}")

### 2.5 Performance evaluation and comparison

In [ ]:
print("=== DECISION TREE (TUNED) ===")
_, dt_acc = evaluate_model(best_dt, X_test, y_test, model_name="Food DT", 
                           save_cm_path="../figures/food_dt_cm.png", class_names=class_names)

print("=== NEURAL NETWORK ===")
_, mlp_acc = evaluate_model(mlp, X_test, y_test, model_name="Food MLP", 
                            save_cm_path="../figures/food_mlp_cm.png", class_names=class_names)